### Code Description:

- This code processes allocation data to calculate the optimal allocation of products for different growth rates.
- It generates pie charts to visualize the allocations and graphs to show the country's ECI growth over time for each growth rate. 
- Additionally, it creates an Excel file that provides allocation data for the country.


#### TO DO:


In [27]:
# TO FIX - PORTFOLIO TABLE EXPORT (take from MD old 3 https://colab.research.google.com/drive/1xq_vhnvKl4fIz0I-LflZXWYPD5gsymo3?usp=sharing)

# TO DO Modify country here
country = 'Colombia'

root_path = 'drive/My Drive/ECI Research/Fall 2022/Data/2019-2021 cleaned/'

path = 'drive/My Drive/New_result_update/'

In [2]:
# After all code is ran, click on the menu on the folder icon to side on the left 
# download "tableau [country] 2021.xls"
# will give a portfolio for the country

# "Italy ECI Growth Optimization Result.png" and "Italy ECI Increment.png" are the graphs produced

#### Import packages

Install xlwt

In [3]:
!pip install xlwt

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.0/100.0 kB 2.4 MB/s eta 0:00:00


Install the kaleido package for exporting images

In [4]:
!pip install -U kaleido

#needed to downgrade plotly for kaleido to work; the code seems to run fine with the lower version,
#but let me know if we run into issues
!pip install plotly==5.3.1

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 10.1 MB/s eta 0:00:00
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.9/23.9 MB 9.0 MB/s eta 0:00:00
  Attempting uninstall: plotly
    Found existing installation: plotly 5.13.1
    Uninstalling plotly-5.13.1:
      Successfully uninstalled plotly-5.13.1


In [5]:
import io
import os
import csv
import statsmodels.api as sm
import math
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import operator
import xlwt
import tqdm
import sys, os
import multiprocessing
import plotly.graph_objects as go
import plotly.graph_objects as go
import pickle

from plotly.offline import plot
from plotly.subplots import make_subplots
from xlwt import Workbook
from functools import partial
from scipy import linalg as LA
import copy

In [6]:
from functools import partial
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pickle

In [7]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

Mounted at /content/drive/


In [28]:
product_list = ['Animals and By-products',
                'Vegetables, Fruits and Plants',
                'Food and Preparations',
                'Energy Drilling and Mining',
                'Chemicals',
                'Chemical Manufacturing',
                'Raw Manufacturing',
                'Wood Products',
                'Clothing and Accessories',
                'Textile',
                'Metals',
                'Machinery and Construction',
                'Transportation',
                'Weapons',
                'Daily Instruments',
                'Precious Stones',
                'High-complexity Products',
                'Services and Utilities']

In [29]:
num_to_products = {1: 'Animals and By-products',
                   2: 'Vegetables, Fruits and Plants',
                   3: 'Food and Preparations',
                   4: 'Energy Drilling and Mining',
                   5: 'Chemicals',
                   6: 'Chemical Manufacturing',
                   7: 'Raw Manufacturing',
                   8: 'Wood Products',
                   9: 'Clothing and Accessories',
                   10: 'Textile',
                   11: 'Metals',
                   12: 'Machinery and Construction',
                   13: 'Transportation',
                   14: 'Weapons',
                   15: 'Daily Instruments',
                   16: 'Precious Stones',
                   17: 'High-complexity Products',
                   18: 'Services and Utilities'}

#### Util

In [30]:
def cal_allocation(allocations, percentages, indices, num_to_products):

    all_total = []
    all_allocation_percentage_dict = []
    all_allocation_dict = []

    for i in range(4):
        all_total.append(np.max(list(allocations[i].values())))



    for i in range(4):

        allocation_percentage_dict = {}

        for iter, index in enumerate(indices[i]):
            if index in list(allocation_percentage_dict.keys()):
              allocation_percentage_dict[index] += percentages[i][iter+1] -\
                                                            percentages[i][iter]
            else:
              allocation_percentage_dict[index] = percentages[i][iter+1] -\
                                                            percentages[i][iter]
        all_allocation_percentage_dict.append(allocation_percentage_dict)

    for i in range(4):

        allocation_dict = {}
        for key, value in all_allocation_percentage_dict[i].items():

            allocation_dict[num_to_products[key]] = value * all_total[i]
        
        all_allocation_dict.append(allocation_dict)

    return all_allocation_dict

#### Pie plots

In [31]:
def pie_plot(country, allocations):

    labels = []
    for i in range(len(allocations)):
        allocations[i] = {x:y for x,y in allocations[i].items() if y!=0}
        labels.append(list(allocations[i].keys()))
    specs = [[{'type':'domain'}, {'type':'domain'}, {'type':'domain'}, {'type':'domain'}]]
    fig = make_subplots(rows=1, cols=4, specs = specs)
    fig.add_trace(go.Pie(labels=labels[0], values=np.asarray(list(allocations[0].values()))/np.max(list(allocations[0].values())), 
                         name="5% growth"), 1, 1)
    fig.add_trace(go.Pie(labels=labels[1], values=np.asarray(list(allocations[1].values()))/np.max(list(allocations[1].values())), 
                         name="10% growth"), 1, 2)
    fig.add_trace(go.Pie(labels=labels[2], values=np.asarray(list(allocations[2].values()))/np.max(list(allocations[2].values())), 
                         name="15% growth"), 1, 3)
    fig.add_trace(go.Pie(labels=labels[3], values=np.asarray(list(allocations[3].values()))/np.max(list(allocations[3].values())), 
                         name="20% growth"), 1, 4)
    
    # Use `hole` to create a donut-like pie chart
    fig.update_traces(hole=.4, hoverinfo="label+percent+name")
    
    fig.update_layout(
        title_text=f"{country} ECI Growth Optimization Result",
        # Add annotations in the center of the donut pies.
        font=dict(
        family="Times New Roman"),
        annotations=[dict(text='5', x=0.095, y=0.5, font_size=20, showarrow=False),
                     dict(text='10', x=0.37, y=0.5, font_size=20, showarrow=False),
                     dict(text='15', x=0.63, y=0.5, font_size=20, showarrow=False),
                     dict(text='20', x=0.91, y=0.5, font_size=20, showarrow=False)],
        width=1000, height=500)
    #fig.update_layout(margin=dict(t=50, b=0, l=0, r=0))
    
    fig.show()
    
    # Save the plot as a PNG image
    fig.write_image(str(country) + " ECI Growth Optimization Result" + ".png", engine="kaleido")


Plot for ECI vs time for different growth rates

In [32]:
import plotly.graph_objects as go

def eic_growth_plot(country, percentages, eci_increment, indices):
    
    data = []
    for i in range(len(percentages)):
        trace = go.Scatter(x=percentages[i],
                           y=eci_increment[i],
                           name=f"{(i+1)*5}% growth",
                           line=dict(width=2),
                           marker=dict(size=6))
        data.append(trace)

    layout = go.Layout(
        xaxis=dict(title="Percentages"),
        title_text=f"{country} ECI Growth",
        font=dict(family="Times New Roman"),
        legend=dict(
            traceorder="reversed"
        ),
    )

    fig = go.Figure(data=data, layout=layout)

    fig.update_layout(width=930, height=350,
                      yaxis_title="ECI",
                      title_text=f"{country} ECI Increment",
                      showlegend=True)

    fig.show()

    # Save the plot as a PNG image
    fig.write_image(str(country) + " ECI Increment" + ".png", engine="kaleido")



In [33]:
#BELOW: Old code that plotted the ECI increments along separate x-axes for the different growth rates


# def eic_growth_plot(country, percentages, eci_increment, indices):

#     trace1 = go.Scatter(x = percentages[0], 
#                           y = eci_increment[0], name="5% growth",  
#                         line=dict(width=2),marker=dict(size=6))
                      
#     trace2 = go.Scatter(x = percentages[1], xaxis="x2",
#                           y = eci_increment[1], name="10% growth",  
#                         line=dict(width=2), marker=dict(size=6))

#     trace3 = go.Scatter(x = percentages[2], xaxis="x3",
#                           y = eci_increment[2], name="15% growth",  
#                         line=dict(width=2), marker=dict(size=6))

#     trace4 = go.Scatter(x = percentages[3], xaxis="x4",
#                           y = eci_increment[3], name="20% growth",  
#                         line=dict(width=2), marker=dict(size=6))

#     data = [trace1, trace2, trace3, trace4]

#     layout = go.Layout(
#         xaxis=dict(
#             domain=[0, 0.25]
#         ),
#         legend=dict(
#             traceorder="reversed"
#         ),
#         xaxis2=dict(
#             domain=[0.25, 0.5]
#         ),
#         xaxis3=dict(
#             domain=[0.5, 0.75]
#         ),
#         xaxis4=dict(
#             domain=[0.75, 1]
#         ),
#         title_text=f"{country} ECI Growth",
#         font=dict(
#         family="Times New Roman"),        
#     )

#     fig = go.Figure(data=data, layout=layout)

#     fig['layout'].update(
#         annotations=[
                    
#         dict(
#             x=(all_percentages[j][i-1] + all_percentages[j][i])/2, y=(eci_increment[j][i-1] + eci_increment[j][i])/2, # annotation point
#             xref=f'x{j+1}', 
#             text=f'{indices[j][i-1]}',
#             showarrow=False,
#             xshift=-6,
#             yshift=10,
#         ) for j in range(4) for i in range(1, len(eci_increment[j]))])

#     fig.update_layout(width=930, height=350,
#                       yaxis_title="ECI",
#                       title_text=f"{country} ECI Increment",
#                       showlegend=True)
    
#     fig.show()

#     # Save the plot as a PNG image
#     fig.write_image(str(country) + " ECI Increment" + ".png", engine="kaleido")

In [34]:
#to do (not needed)

year = 2021
num_share = [30 * i for i in range(1,5)]
budget_percentage = [5, 10, 15,20]

In [35]:
allocation = []

for j in range(4):
    with open(path + f'{country}_{year}_percentage{budget_percentage[j]}_part{num_share}.p', 'rb') as fp:
        allocation.append(pickle.load(fp))

In [36]:
allocations = [i[0] for i in allocation]
all_percentages = [i[1] for i in allocation]
all_eci_increments = [i[2] for i in allocation]
all_indices = [i[4] for i in allocation]

all_allocations = cal_allocation(allocations, all_percentages, all_indices, 
                                 num_to_products)

In [37]:
# TO FIX
pie_plot(country, all_allocations)
eic_growth_plot(country, all_percentages, all_eci_increments, all_indices)

#### Tableau print

In [38]:
def write_tableu(country, year, all_allocations, all_eci_increments, budgets, product_list):

    all_products = []

    for i in range(4):

      all_products.extend(list(all_allocations[i].keys()))

    all_products = set(all_products)

    tableau_products = []

    for product in product_list:
        if product in all_products:
            tableau_products.append(product)

    col_num=0
    book = Workbook()
    sheet1 = book.add_sheet('Sheet 1')
    st = xlwt.easyxf('pattern: pattern solid;')
    st.pattern.pattern_fore_colour = 1
    sheet1.write(0, col_num, 'Heuristic Export Growth', st)

    for i in range(0,len(tableau_products)):

          st = xlwt.easyxf('pattern: pattern solid;')
          st.pattern.pattern_fore_colour = 1
          sheet1.write(i+1, col_num, tableau_products[i], st)
          
    st = xlwt.easyxf('pattern: pattern solid;')
    st.pattern.pattern_fore_colour = 1
    sheet1.write(len(tableau_products)+1, col_num, 'Increased ECI', st)

    for iter, budget in enumerate(budgets):

        col_num += 1
        sheet1.write(0, col_num, budget, st)
        for j, product in enumerate(tableau_products):
            try:
              
              st = xlwt.easyxf('pattern: pattern solid;')
              st.pattern.pattern_fore_colour = 1
              sheet1.write(j+1, col_num, all_allocations[iter][product], st)
            except:

              st = xlwt.easyxf('pattern: pattern solid;')
              st.pattern.pattern_fore_colour = 1
              sheet1.write(j+1, col_num, 0, st)
        
        st = xlwt.easyxf('pattern: pattern solid;')
        st.pattern.pattern_fore_colour = 1
        sheet1.write(j+2, col_num, all_eci_increments[iter][-1], st)

    print(f'{country} {year} completed')
    book.save(f'tableau {country} {year}.xls')

    return

In [39]:
#country = 'Plurinational State of Bolivia'

#year = 2013
num_share = [30*i for i in range(1,5)]
budget_percentage = [5, 10, 15, 20]
budgets = ['5%', '10%', '15%', '20%']

In [40]:
allocation = []

for j in range(4):
  
      with open(path + f'{country}_{year}_percentage{budget_percentage[j]}_part{num_share}.p', 'rb') as fp:
            allocation.append(pickle.load(fp))

In [41]:
allocations = [i[0] for i in allocation]
all_percentages = [i[1] for i in allocation]
all_eci_increments = [i[2] for i in allocation]
all_indices = [i[4] for i in allocation]

all_allocations = cal_allocation(allocations, all_percentages, all_indices, 
                                 num_to_products)

In [42]:
# TO DO: 
write_tableu(country, year, all_allocations, all_eci_increments, budgets, product_list)

#click on folder icon to side on the left after running this and download .xls
# will give a portfolio for the countrry 

Colombia 2021 completed


In [43]:
end

NameError: ignored

#### Test --DO NOT RUN FOR REPORTS

In [ ]:
country = 'Canada'
percentages = all_percentages
eci_increment = all_eci_increments
indices = all_indices

In [ ]:
trace1 = go.Scatter(x = percentages[0], 
                      y = eci_increment[0], name="5% growth",  
                    line=dict(width=4),marker=dict(size=10))
                  
trace2 = go.Scatter(x = percentages[1], xaxis="x2",
                      y = eci_increment[1], name="10% growth",  
                    line=dict(width=4), marker=dict(size=10))

trace3 = go.Scatter(x = percentages[2], xaxis="x3",
                      y = eci_increment[2], name="15% growth",  
                    line=dict(width=4), marker=dict(size=10))

trace4 = go.Scatter(x = percentages[3], xaxis="x4",
                      y = eci_increment[3], name="20% growth",  
                    line=dict(width=4), marker=dict(size=10))

data = [trace1, trace2, trace3, trace4]

layout = go.Layout(
    xaxis=dict(
        domain=[0, 0.25]
    ),
    legend=dict(
        traceorder="reversed"
    ),
    xaxis2=dict(
        domain=[0.25, 0.5]
    ),
    xaxis3=dict(
        domain=[0.5, 0.75]
    ),
    xaxis4=dict(
        domain=[0.75, 1]
    ),
    title_text=f"{country} ECI Growth",
    font=dict(
    family="Times New Roman"),        
)

fig = go.Figure(data=data, layout=layout)

fig['layout'].update(
    annotations=[
                
    dict(
        x=(all_percentages[j][i-1] + all_percentages[j][i])/2, y=(eci_increment[j][i-1] + eci_increment[j][i])/2, # annotation point
        xref=f'x{j+1}', 
        text=f'{indices[i-1]}',
        showarrow=False,
        yshift=10,
    ) for j in range(4) for i in range(1, len(eci_increment[j]))])

fig.update_layout(width=930, height=350,
                  yaxis_title="ECI",
                  title_text=f"{country} ECI Increment",
                  showlegend=True)

fig.show()

In [ ]:
for j in range(4):
   for i in range(1, len(eci_increment[j])):
                
        dict(
            x=(all_percentages[j][i-1] + all_percentages[j][i])/2, y=(eci_increment[j][i-1] + eci_increment[j][i])/2, # annotation point
            xref=f'x{j+1}', 
            text=f'{indices[j][i-1]}',
            showarrow=False,
            yshift=10,
        ) 

In [ ]:
trace1 = go.Scatter(mode="lines+markers+text",
                    x = list(range(0, len(eci_increment[0]))), 
                    y = eci_increment[0], name="5% growth",  
                    line=dict(width=4),marker=dict(size=10))
                      
trace2 = go.Scatter(x = list(range(0, len(eci_increment[1]))), xaxis="x2",
                      y = eci_increment[1], name="10% growth",  
                    line=dict(width=4), marker=dict(size=10))

trace3 = go.Scatter(x = list(range(0, len(eci_increment[2]))), xaxis="x3",
                      y = eci_increment[2], name="15% growth",  
                    line=dict(width=4), marker=dict(size=10))

trace4 = go.Scatter(x = list(range(0, len(eci_increment[3]))), xaxis="x4",
                      y = eci_increment[3], name="20% growth",  
                    line=dict(width=4), marker=dict(size=10))

data = [trace1, trace2, trace3, trace4]

layout = go.Layout(
    xaxis=dict(
        domain=[0, 0.25]
    ),
    legend=dict(
        traceorder="reversed"
    ),
    xaxis2=dict(
        domain=[0.25, 0.5]
    ),
    xaxis3=dict(
        domain=[0.5, 0.75]
    ),
    xaxis4=dict(
        domain=[0.75, 1]
    ),
    title_text=f"{country} ECI Growth",
    font=dict(
    family="Times New Roman"),        
)

fig = go.Figure(data=data, layout=layout)

fig['layout'].update(
    annotations=[
                 
    dict(
        x=i+0.5, y=(eci_increment[j][i] + eci_increment[j][i+1])/2, # annotation point
        xref=f'x{j+1}', 
        text=f'{i}',
        showarrow=False,
        #arrowhead=7,
        #ax=10,
        #ay=70.
        #xshift=-5,
        yshift=10,
    ) for j in range(4) for i in range(len(eci_increment[j])-1)])

'''
for i in range(len(eci_increment[0])-1):

      fig.add_annotation(x=i+0.5, y=(eci_increment[j][i] + eci_increment[j][i+1])/2,
                  text= f'{i}',
                  showarrow=False,
                  yshift=10)
'''     
fig.update_layout(width=930, height=350,
                  yaxis_title="ECI",
                  title_text=f"{country} ECI Increment",
                  showlegend=True)
fig.show()

In [ ]:
for j in range(4):
  for i in range(len(eci_increment[j])-1):
    dict(
          x=i+0.5, y=(eci_increment[j][i] + eci_increment[j][i+1])/2, # annotation point
          xref=f'x{j+1}', 
          text=f'{i}',
          #showarrow=True,
          #arrowhead=7,
          #ax=10,
          #ay=70
          yshift=10
      )  

In [ ]:
layout

In [ ]:
["%.2f" % eci for eci in eci_increment[0]]

In [ ]:
all_products = []

for i in range(4):

  all_products.extend(list(all_allocations[i].keys()))

all_products = set(all_products)

In [ ]:
all_products

In [ ]:
tableau_products = []

for product in product_list:
    if product in all_products:
        tableau_products.append(product)

In [ ]:
col_num=0

In [ ]:
budgets = ['5%', '10%', '15%', '20%']

In [ ]:
col_num=0
book = Workbook()
sheet1 = book.add_sheet('Sheet 1')
st = xlwt.easyxf('pattern: pattern solid;')
st.pattern.pattern_fore_colour = 1
sheet1.write(0, col_num, 'Heuristic Export Growth', st)

for i in range(0,len(tableau_products)):

      st = xlwt.easyxf('pattern: pattern solid;')
      st.pattern.pattern_fore_colour = 1
      sheet1.write(i+1, col_num, tableau_products[i], st)
      
st = xlwt.easyxf('pattern: pattern solid;')
st.pattern.pattern_fore_colour = 1
sheet1.write(len(tableau_products)+1, col_num, 'Increased ECI', st)

for iter, budget in enumerate(budgets):

    col_num += 1
    sheet1.write(0, col_num, budget, st)
    for j, product in enumerate(tableau_products):
        try:
           
           st = xlwt.easyxf('pattern: pattern solid;')
           st.pattern.pattern_fore_colour = 1
           sheet1.write(j+1, col_num, all_allocations[iter][product], st)
        except:

           st = xlwt.easyxf('pattern: pattern solid;')
           st.pattern.pattern_fore_colour = 1
           sheet1.write(j+1, col_num, 0, st)
    
    st = xlwt.easyxf('pattern: pattern solid;')
    st.pattern.pattern_fore_colour = 1
    sheet1.write(j+2, col_num, all_eci_increments[iter][-1], st)

print(f'{country} completed')
book.save(f'tableau.xls')